# GeoClip Zero-Shot Baseline

Evaluate pretrained GeoClip on the MMlandmarks query set **without fine-tuning**.
The model embeds query ground images and gallery GPS coordinates into a shared
512-dim space, then retrieves the nearest GPS by cosine similarity.

**Gallery:** configurable via `gallery.source` in [configs/geoclip_baseline.yaml](../../configs/geoclip_baseline.yaml):
- `"paper"` (default, 100,539 coords = 99,539 index-satellite + 1,000 query-landmark GPS) — matches the camera-ready MML paper Sec 5.2 protocol. Reproduces the 21.37 % @1 km row of Table 3. Because every query's GT GPS is in the gallery, this is an **upper bound**.
- `"index"` (99,539 coords) — index-satellite only. Honest in-the-wild result (~6.67 % @1 km). Per the paper author: *"21 % is a geolocalization upper limit, 6.67 % is more realistic in the wild."*

**Queries:** 18,688 query ground images (multiple images per landmark, each scored against the landmark's ground-truth GPS).

**Metric:** Accuracy @ {1, 25, 200, 750, 2500} km (Haversine distance).

## 1. Setup

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import yaml

plt.rcParams.update({"figure.dpi": 120})

# Load config
with open("../../configs/geoclip_baseline.yaml") as f:
    cfg = yaml.safe_load(f)

DATA_ROOT = Path("../../") / cfg["data"]["root"]
assert DATA_ROOT.exists(), f"DATA_ROOT not found: {DATA_ROOT}"

device = cfg["inference"]["device"] if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Data root: {DATA_ROOT.resolve()}")

Device: cuda
Data root: /dtu/blackhole/02/137570/MML


## 2. Load Model

In [2]:
from mmgeo.geolocalizations.geoclip.geoclip_baseline import (
    GeoClipBaseline,
    load_gallery_coords,
    load_query_data3,
    NewGeoClipBaseline
)
from mmgeo.geolocalizations.geoclip.evaluate import (
    accuracy_at_thresholds,
    median_error,
    haversine,
)

baseline = NewGeoClipBaseline(device=device,transformer=True)
#we import model from models/12new_geoclip.pth and put it into baseline.model
model_path = Path("../../models/12new_geoclip.pth")
assert model_path.exists(), f"Model not found: {model_path}"
state_dict = torch.load(model_path, map_location=device)
baseline.model.load_state_dict(state_dict)

total_params = sum(p.numel() for p in baseline.model.parameters())
trainable_params = sum(p.numel() for p in baseline.model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

/zhome/79/0/186934/Multimodal-Geo-Spatial-Learning/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 9734.36it/s]

/zhome/79/0/186934/Multimodal-Geo-Spatial-Learning/.venv/lib/python3.11/site-packages/geoclip/model/location_encoder.py:57: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sel

/zhome/79/0/186934/Multimodal-Geo-Spatial-Learning/.venv/lib/python3.11/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
/tmp/ipykernel_3253629/2145309542.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.s

Total parameters: 450,660,354
Trainable parameters: 23,042,305


## 3. Build GPS Gallery

In [3]:
# Load both galleries up-front; we rebuild per-source before each inference pass.
GALLERY_SOURCES = ["paper", "index"]
galleries = {}
for source in GALLERY_SOURCES:
    coords = load_gallery_coords(DATA_ROOT, source=source)
    galleries[source] = coords
    print(
        f"source={source!r:>8}: {len(coords):>6,} GPS points · "
        f"lat [{coords[:, 0].min():.2f}, {coords[:, 0].max():.2f}] · "
        f"lon [{coords[:, 1].min():.2f}, {coords[:, 1].max():.2f}]"
    )


source= 'paper': 100,539 GPS points · lat [20.00, 49.03] · lon [-155.89, -66.93]
source= 'index': 99,539 GPS points · lat [20.00, 49.03] · lon [-155.89, -66.93]


## 4. Load Query Data

In [4]:
thing = 1
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (1000, 1000, 1000)

[paper] build_gallery (100,539 GPS points)…


[paper] 1000 predictions in 29.8s

[index] build_gallery (99,539 GPS points)…


[index] 1000 predictions in 28.2s
 Threshold (km) paper (%) index (%)
              1      7.30      1.50
             25     22.90     20.40
            200     41.80     39.30
            750     70.90     69.50
           2500     93.20     92.30

 paper: median error 290.5 km · mean 660.7 km
 index: median error 323.8 km · mean 705.6 km


In [5]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (1902, 951, 1902)

[paper] build_gallery (100,539 GPS points)…


[paper] 951 predictions in 53.8s

[index] build_gallery (99,539 GPS points)…


[index] 951 predictions in 53.2s
 Threshold (km) paper (%) index (%)
              1     12.09      2.73
             25     28.81     26.60
            200     52.37     51.00
            750     80.97     79.50
           2500     96.85     96.32

 paper: median error 172.8 km · mean 443.8 km
 index: median error 191.3 km · mean 479.4 km


In [6]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (2583, 861, 2583)

[paper] build_gallery (100,539 GPS points)…


[paper] 861 predictions in 72.8s

[index] build_gallery (99,539 GPS points)…


[index] 861 predictions in 72.5s
 Threshold (km) paper (%) index (%)
              1     15.21      3.25
             25     36.24     32.98
            200     59.23     57.61
            750     84.79     83.39
           2500     97.44     96.86

 paper: median error 115.3 km · mean 366.9 km
 index: median error 128.7 km · mean 405.4 km


In [7]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (3104, 776, 3104)

[paper] build_gallery (100,539 GPS points)…


[paper] 776 predictions in 86.9s

[index] build_gallery (99,539 GPS points)…


[index] 776 predictions in 86.3s
 Threshold (km) paper (%) index (%)
              1     18.56      4.51
             25     39.82     35.70
            200     62.89     60.70
            750     85.70     83.76
           2500     97.55     97.04

 paper: median error 95.8 km · mean 339.7 km
 index: median error 119.3 km · mean 384.7 km


In [8]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (3490, 698, 3490)

[paper] build_gallery (100,539 GPS points)…


[paper] 698 predictions in 99.6s

[index] build_gallery (99,539 GPS points)…


[index] 698 predictions in 98.6s
 Threshold (km) paper (%) index (%)
              1     20.20      5.01
             25     41.98     37.39
            200     65.19     62.89
            750     87.39     85.67
           2500     97.85     96.99

 paper: median error 72.4 km · mean 309.7 km
 index: median error 101.6 km · mean 366.7 km


In [9]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (3738, 623, 3738)

[paper] build_gallery (100,539 GPS points)…


[paper] 623 predictions in 105.8s

[index] build_gallery (99,539 GPS points)…


[index] 623 predictions in 105.6s
 Threshold (km) paper (%) index (%)
              1     21.99      5.62
             25     45.26     40.61
            200     68.22     65.81
            750     88.92     86.84
           2500     98.07     97.27

 paper: median error 52.9 km · mean 278.1 km
 index: median error 79.5 km · mean 336.6 km


In [10]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (4011, 573, 4011)

[paper] build_gallery (100,539 GPS points)…


[paper] 573 predictions in 114.8s

[index] build_gallery (99,539 GPS points)…


[index] 573 predictions in 114.5s
 Threshold (km) paper (%) index (%)
              1     22.69      5.76
             25     46.77     41.36
            200     68.59     65.45
            750     88.31     85.17
           2500     98.25     97.03

 paper: median error 52.9 km · mean 280.9 km
 index: median error 88.2 km · mean 360.7 km


In [11]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (4184, 523, 4184)

[paper] build_gallery (100,539 GPS points)…


[paper] 523 predictions in 118.9s

[index] build_gallery (99,539 GPS points)…


[index] 523 predictions in 118.7s
 Threshold (km) paper (%) index (%)
              1     24.86      6.31
             25     49.14     42.83
            200     70.55     66.35
            750     89.10     85.66
           2500     98.28     96.56

 paper: median error 30.4 km · mean 270.8 km
 index: median error 72.1 km · mean 369.9 km


In [12]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (4419, 491, 4419)

[paper] build_gallery (100,539 GPS points)…


[paper] 491 predictions in 125.9s

[index] build_gallery (99,539 GPS points)…


[index] 491 predictions in 124.3s
 Threshold (km) paper (%) index (%)
              1     25.05      5.70
             25     48.27     42.57
            200     69.86     65.99
            750     89.82     86.35
           2500     98.17     96.33

 paper: median error 33.0 km · mean 265.0 km
 index: median error 74.0 km · mean 369.1 km


In [13]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (4550, 455, 4550)

[paper] build_gallery (100,539 GPS points)…


[paper] 455 predictions in 130.5s

[index] build_gallery (99,539 GPS points)…


[index] 455 predictions in 129.6s
 Threshold (km) paper (%) index (%)
              1     27.91      6.37
             25     52.75     46.59
            200     72.09     69.01
            750     91.43     87.91
           2500     98.68     97.14

 paper: median error 16.6 km · mean 231.1 km
 index: median error 47.7 km · mean 324.7 km


In [14]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (4631, 421, 4631)

[paper] build_gallery (100,539 GPS points)…


[paper] 421 predictions in 130.9s

[index] build_gallery (99,539 GPS points)…


[index] 421 predictions in 130.5s
 Threshold (km) paper (%) index (%)
              1     29.45      6.65
             25     53.21     46.32
            200     71.02     67.46
            750     90.97     87.41
           2500     98.34     96.20

 paper: median error 13.6 km · mean 247.8 km
 index: median error 47.7 km · mean 354.1 km


In [15]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (4752, 396, 4752)

[paper] build_gallery (100,539 GPS points)…


[paper] 396 predictions in 134.8s

[index] build_gallery (99,539 GPS points)…


[index] 396 predictions in 133.8s
 Threshold (km) paper (%) index (%)
              1     29.29      6.31
             25     54.29     47.22
            200     72.47     68.43
            750     91.92     88.13
           2500     98.23     95.96

 paper: median error 13.1 km · mean 235.5 km
 index: median error 42.4 km · mean 350.5 km


In [16]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (4966, 382, 4966)

[paper] build_gallery (100,539 GPS points)…


[paper] 382 predictions in 141.2s

[index] build_gallery (99,539 GPS points)…


[index] 382 predictions in 140.8s
 Threshold (km) paper (%) index (%)
              1     29.84      6.02
             25     54.97     46.60
            200     72.25     68.59
            750     92.15     89.01
           2500     98.17     96.34

 paper: median error 9.6 km · mean 235.6 km
 index: median error 45.5 km · mean 334.2 km


In [17]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (5096, 364, 5096)

[paper] build_gallery (100,539 GPS points)…


[paper] 364 predictions in 145.4s

[index] build_gallery (99,539 GPS points)…


[index] 364 predictions in 145.2s
 Threshold (km) paper (%) index (%)
              1     31.32      6.87
             25     55.77     47.25
            200     73.63     69.51
            750     92.31     87.91
           2500     97.80     95.60

 paper: median error 8.2 km · mean 243.3 km
 index: median error 35.6 km · mean 362.7 km


In [18]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (5310, 354, 5310)

[paper] build_gallery (100,539 GPS points)…


[paper] 354 predictions in 147.6s

[index] build_gallery (99,539 GPS points)…


[index] 354 predictions in 146.6s
 Threshold (km) paper (%) index (%)
              1     33.05      7.63
             25     55.65     47.46
            200     74.58     70.34
            750     91.53     87.29
           2500     97.74     95.48

 paper: median error 7.9 km · mean 247.3 km
 index: median error 35.4 km · mean 371.8 km


In [19]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (5328, 333, 5328)

[paper] build_gallery (100,539 GPS points)…


[paper] 333 predictions in 149.3s

[index] build_gallery (99,539 GPS points)…


[index] 333 predictions in 148.8s
 Threshold (km) paper (%) index (%)
              1     33.33      8.41
             25     55.56     47.75
            200     73.87     69.37
            750     90.99     85.89
           2500     97.30     94.59

 paper: median error 9.5 km · mean 264.8 km
 index: median error 35.0 km · mean 407.0 km


In [20]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (5287, 311, 5287)

[paper] build_gallery (100,539 GPS points)…


[paper] 311 predictions in 147.5s

[index] build_gallery (99,539 GPS points)…


[index] 311 predictions in 147.1s
 Threshold (km) paper (%) index (%)
              1     33.12      9.00
             25     56.27     48.55
            200     75.24     71.38
            750     91.32     86.50
           2500     97.43     94.86

 paper: median error 6.9 km · mean 252.6 km
 index: median error 31.6 km · mean 389.9 km


In [21]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (5328, 296, 5328)

[paper] build_gallery (100,539 GPS points)…


[paper] 296 predictions in 150.4s

[index] build_gallery (99,539 GPS points)…


[index] 296 predictions in 149.9s
 Threshold (km) paper (%) index (%)
              1     34.46     10.14
             25     57.09     48.99
            200     75.68     70.95
            750     92.57     87.84
           2500     98.31     95.27

 paper: median error 5.9 km · mean 215.3 km
 index: median error 30.6 km · mean 366.4 km


In [22]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1

Query images: (5377, 283, 5377)

[paper] build_gallery (100,539 GPS points)…


[paper] 283 predictions in 148.8s

[index] build_gallery (99,539 GPS points)…


[index] 283 predictions in 147.1s
 Threshold (km) paper (%) index (%)
              1     34.28      9.89
             25     56.89     49.12
            200     76.33     72.08
            750     92.23     87.28
           2500     98.23     95.41

 paper: median error 6.1 km · mean 218.6 km
 index: median error 29.7 km · mean 365.2 km


In [23]:
image_paths, true_coords, landmark_ids = load_query_data3(DATA_ROOT, imagesperlandmark=thing)
print(f"Query images: {len(image_paths),len(true_coords),len(landmark_ids)}")
#print(f"First 20 landmark ids: {landmark_ids[:20]}")
#print(f"True coords: {true_coords[0]} (landmark_id={landmark_ids[0]})")
#print(f"Unique landmarks: {len(set(landmark_ids))}")
#continue

'''
#how many of each landmark
x = pd.Series(landmark_ids).value_counts().value_counts().sort_index()
#every value should also have the sum of the values after it
x = x.sort_index(ascending=False).cumsum().sort_index()
plt.plot(x.index, x.values, marker="o")
#xticks should be 1,5,10,15,20
plt.xticks([1, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100])
'''

# Run inference once per gallery source. Model weights stay the same; only the
# gallery (GPS embeddings cached inside baseline) changes between runs.
predictions = {}
for source, coords in galleries.items():
    print(f"\n[{source}] build_gallery ({len(coords):,} GPS points)…")
    baseline.build_gallery(coords)
    t0 = time.time()
    predictions[source] = baseline.predict_batch(
        image_paths, landmark_ids, batch_size=cfg["inference"]["batch_size"]
    )
    print(f"[{source}] {len(predictions[source])} predictions in {time.time() - t0:.1f}s")


    thresholds = cfg["evaluation"]["thresholds_km"]
true_lat, true_lon = true_coords[:, 0], true_coords[:, 1]

'''
#truecoords is 18k and predictions is 1k, so we copy predicitons to match the length of truecoords for evaluation purposes.this is done using landmark ids
_, inverse_indices, landmark_counts = np.unique(landmark_ids, return_inverse=True, return_counts=True)
for source in GALLERY_SOURCES:
    predictions[source] = np.repeat(predictions[source], landmark_counts, axis=0)
'''

results_by_source = {}
distances_by_source = {}
for source, preds in predictions.items():
    p_lat, p_lon = preds[:, 0], preds[:, 1]
    results_by_source[source] = accuracy_at_thresholds(
        p_lat, p_lon, true_lat, true_lon, thresholds
    )
    distances_by_source[source] = haversine(p_lat, p_lon, true_lat, true_lon)

# Side-by-side comparison table: rows = thresholds, columns = sources.
rows = []
for t in thresholds:
    row = {"Threshold (km)": t}
    for source in GALLERY_SOURCES:
        row[f"{source} (%)"] = f"{results_by_source[source][t] * 100:.2f}"
    rows.append(row)
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print()
for source in GALLERY_SOURCES:
    d = distances_by_source[source]
    print(f"{source:>6}: median error {np.median(d):.1f} km · mean {d.mean():.1f} km")
thing += 1
print(thing)

Query images: (5480, 274, 5480)

[paper] build_gallery (100,539 GPS points)…


[paper] 274 predictions in 153.1s

[index] build_gallery (99,539 GPS points)…


[index] 274 predictions in 153.3s
 Threshold (km) paper (%) index (%)
              1     34.31     10.58
             25     56.57     48.91
            200     76.64     72.26
            750     91.97     87.59
           2500     97.81     95.26

 paper: median error 6.1 km · mean 223.2 km
 index: median error 30.6 km · mean 357.4 km
21


## 5. Run Inference

## 6. Evaluate

## 7. Visualizations

## 8. Summary for Zero Shot

Zero-shot GeoClip on MMlandmarks, 18,688 query ground images, V100. Single HPC submit
runs both gallery sources back-to-back.

### `gallery.source: paper` — paper protocol (index + query = 100,539 GPS)

Reproduces the MML paper's Table 3 off-the-shelf GeoCLIP row within rounding. Because
every query's GT GPS sits in the gallery, this is an **upper bound** on achievable
performance, not an in-the-wild number.

| Threshold (km) | Accuracy (%) |
|---------------:|-------------:|
| 1              | 21.35        |
| 25             | 36.44        |
| 200            | 48.61        |
| 750            | 71.41        |
| 2500           | 91.52        |

- **Median error:** 225.2 km
- **Mean error:** 674.6 km

### `gallery.source: index` — honest in-the-wild (99,539 GPS, no query leakage)

Same model, query GT GPS removed from the gallery. Measures what off-the-shelf GeoCLIP
can actually do on US landmark localization without gallery leakage. Per Oskar
Kristoffersen (first author): *"21 % is a geolocalization upper limit, 6.67 % is more
realistic in the wild."*

| Threshold (km) | Accuracy (%) |
|---------------:|-------------:|
| 1              |  6.67        |
| 25             | 28.79        |
| 200            | 44.48        |
| 750            | 69.07        |
| 2500           | 91.07        |

- **Median error:** 294.3 km
- **Mean error:** 724.2 km

### Paper contrast

| Method | Dataset | Gallery | @1 km | @25 km | @200 km | @750 km | @2500 km |
|---|---|---:|---:|---:|---:|---:|---:|
| GeoClip (own paper) | Im2GPS3k (global) | 100k | 14.11 | 34.47 | 50.65 | 69.67 | 83.82 |
| Off-shelf GeoClip (MML paper) | MMlandmarks (US) | 101k (index+query) | **21.37** | **36.44** | 48.57 | 71.45 | 91.50 |
| **Ours (`paper`)** | MMlandmarks (US) | 101k (index+query) | **21.35** | **36.44** | 48.61 | 71.41 | 91.52 |
| **Ours (`index`)** | MMlandmarks (US) | 100k (index only) | **6.67** | **28.79** | 44.48 | 69.07 | 91.07 |

We reproduce the MML paper row to within 0.02 points at @1 km. The gap between the two
`Ours` rows is the query-leakage effect: including query GT coordinates in the gallery
gives the model a guaranteed-correct candidate to pick, inflating all thresholds.

> An earlier run on the 17,557 train-landmark gallery scored 19.22 % @1 km. That number
> is inflated by cluster-luck — train and query landmarks co-locate in the same tourist
> cities, so the nearest train-landmark GPS is often coincidentally <1 km from a query.
> Not a fair comparison to either of the paper galleries above.

**Next steps (Phase 2):** Fine-tune the Location Encoder and linear image head on the
MMlandmarks train split. Fair improvement lives on top of the `index` baseline
(28.79 % @25 km, not 36.44 %) — the paper gallery's leakage makes it hard to beat by
model changes alone.